## Libraries and functions

In [1]:
import sys
from pathlib import Path

# Add src/ to sys.path regardless of where Jupyter is launched from
for _candidate in [Path().resolve().parent / "src", Path().resolve() / "src"]:
    if _candidate.exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        print(f"Added to sys.path: {_candidate}")
        break

Added to sys.path: C:\Users\loren\Documents\Postdoc\Compressed_sensing\compressed_sensing_bioacoustics\src


In [ ]:
from compress import Compression

import psutil
import time
import os
import threading
from codecarbon import EmissionsTracker
import datetime
import pandas as pd
import gc
# Run garbage collection
gc.collect()

c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\codecarbon\core\gpu.py:4: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


13

In [3]:
def monitor_memory(stop_event, mem_usage, sampling_interval=0.1):
    """
    Monitor the memory usage of the current process at regular intervals.
    """
    process = psutil.Process(os.getpid())
    while not stop_event.is_set():
        mem = process.memory_info().rss / (1024 ** 2)  # Convert bytes to MB
        mem_usage.append(mem)
        time.sleep(sampling_interval)

In [4]:
def monitor_resources(stop_event, cpu_usage, mem_usage, sampling_interval=0.1):
    """
    Monitor CPU and memory usage of the current process at regular intervals.
    - cpu_usage: list to store CPU usage (%) over time
    - mem_usage: list to store memory usage (MB) over time
    """
    process = psutil.Process(os.getpid())
    
    while not stop_event.is_set():
        # CPU percent since last call (per process)
        cpu = process.cpu_percent(interval=None)
        # Memory usage (RSS in MB)
        mem = process.memory_info().rss / (1024 ** 2)
        
        cpu_usage.append(cpu)
        mem_usage.append(mem)
        
        time.sleep(sampling_interval)

## Compression

In [5]:
#parameters
folder_audio=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Bats\Audio"
folder_compress=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Bats\Compressed_Audio"
folder_tracking=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Bats\tracking"
converter_path="c:/Users/loren/Documents/Postdoc/Compressed_sensing/ffmpeg-master-latest-win64-gpl-shared/ffmpeg-master-latest-win64-gpl-shared/bin/ffmpeg.exe"

method_compression="flac"  #between mp3, acc, ogg, flac
parameter_compression="0" #for mp3 ; 96k/192k/320k, flac;0/6/12, ogg:0/5/10  , aac; 96k/192k/320k

emissions_file = Path(folder_tracking) / f"emissions_output_{method_compression}_{parameter_compression}.csv"

In [6]:
# Initialize compression class
compression = Compression(folder_audio, folder_compress, method_compression, parameter_compression ,converter_path)


In [7]:
# Lists to store metrics
cpu_usage, mem_usage = [], []
sampling_interval = 0.1  # Adjust as needed
stop_event = threading.Event()


# Start resource monitoring in background
monitor_thread = threading.Thread(target=monitor_resources, args=(stop_event,cpu_usage, mem_usage, sampling_interval))
init_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-5]
print(init_time)
monitor_thread.start()

# Start emissions tracking
tracker = EmissionsTracker(output_file=str(emissions_file))
tracker.start()

# Run compression
starting = time.time()
times=compression.compress()
end=time.time()
execution_time_baseline = time.time() - starting


# Stop trackers
tracker.stop()
stop_event.set()
monitor_thread.join()



print(execution_time_baseline)

Exception in thread Thread-16 (monitor_resources):
[codecarbon INFO @ 08:34:59] Codecarbon is taking the configuration from global file: C:\Users\loren\.codecarbon.config
Traceback (most recent call last):
  File "c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\loren\AppData\Local\Temp\ipykernel_17284\1853832791.py", line 7, in monitor_resources
    process = psutil.Process(os.getpid())
                             ^^
NameError: name 'os' is not defined. Did you forget to import 'os'?
[codecarbon WARNING @ 08:34:59] Multiple instances of codecarbon a

2026-05-01 08:34:59.2


[codecarbon WARNING @ 08:35:01] We saw that you have a 13th Gen Intel(R) Core(TM) i7-13650HX but we don't know it. Please contact us.
[codecarbon WARNING @ 08:35:01] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 08:35:01] CPU Model on constant consumption mode: 13th Gen Intel(R) Core(TM) i7-13650HX
[codecarbon WARNING @ 08:35:01] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 08:35:01] [setup] GPU Tracking...
[codecarbon INFO @ 08:35:02] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 08:35:02] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global constant
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 08:35:02] >>> Tracker's metadata:
[codecarbon INFO @ 08:35:02]   Platform system: Windows-11-10.0.2620

compression file : 000000_CISTUGO__SMU03992_20210927_235835.wav
compression file : 000000_CISTUGO__SMU05135_20230225_213710.wav
compression file : 000001_CISTUGO__SMU03992_20210927_234748.wav
compression file : 000001_CISTUGO__SMU03992_20210928_223918.wav
compression file : 000002_CISTUGO__SMU03829_20220709_181150.wav
compression file : 000002_CISTUGO__SMU05135_20230225_213759.wav
compression file : 000003_CISTUGO__SMU03829_20230302_051610.wav
compression file : 000003_CISTUGO__SMU03992_20210928_225648.wav
compression file : 000004_CISTUGO__SMU03992_20210927_051624.wav
compression file : 000004_CISTUGO__SMU03992_20210928_002130.wav
compression file : 000005_CISTUGO__SMU03992_20210927_051557.wav
compression file : 000005_CISTUGO__SMU03992_20210928_041559.wav
compression file : 000006_CISTUGO__SMU05135_20230225_212128.wav
compression file : 000006_CISTUGO__SMU05135_20230225_212306.wav
compression file : 000007_CISTUGO__SMU03992_20210928_223059.wav
compression file : 000007_CISTUGO__SMU05

[codecarbon INFO @ 08:35:20] Energy consumed for RAM : 0.000043 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 08:35:20] Delta energy consumed for CPU with constant : 0.000183 kWh, power : 42.5 W
[codecarbon INFO @ 08:35:20] Energy consumed for All CPU : 0.000183 kWh


...

compression file : 002239_TADAEG__SMU03992_20220707_181451.wav
compression file : 002240_TADAEG__SMU03812_20211001_202448.wav
compression file : 002241_TADAEG__SMU04959_20220713_180323.wav
compression file : 002242_TADAEG__SMU04959_20210928_195837.wav
compression file : 002243_TADAEG__SMU04959_20210929_003838.wav
compression file : 002244_TADAEG__SMU04959_20230303_233834.wav
compression file : 002245_TADAEG__SMU04959_20210927_215058.wav
compression file : 002246_TADAEG__SMU03812_20211004_191407.wav
compression file : 002247_TADAEG__SMU04959_20210927_223545.wav
compression file : 002248_TADAEG__SMU04959_20230301_005457.wav
compression file : 002249_TADAEG__SMU04959_20210927_222650.wav
compression file : 002250_TADAEG__SMU03992_20220706_203616.wav
compression file : 002251_TADAEG__SMU04959_20210929_011055.wav
compression file : 002252_TADAEG__SMU04959_20220714_181520.wav
compression file : 002253_TADAEG__SMU04959_20230228_203910.wav
compression file : 002254_TADAEG__SMU05135_20230226_025

[codecarbon INFO @ 08:45:32] Energy consumed for RAM : 0.001626 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 08:45:32] Delta energy consumed for CPU with constant : 0.000128 kWh, power : 42.5 W
[codecarbon INFO @ 08:45:32] Energy consumed for All CPU : 0.006914 kWh


compression file : 002313_TADAEG__SMU04959_20220714_180313.wav


[codecarbon INFO @ 08:45:33] Energy consumed for all GPUs : 0.000548 kWh. Total GPU Power : 2.7452715539003445 W
[codecarbon INFO @ 08:45:33] 0.009088 kWh of electricity used since the beginning.
[codecarbon WARNING @ 08:45:34] graceful shutdown. Exceptions:
[codecarbon WARNING @ 08:45:34] <class 'Exception'>
Traceback (most recent call last):
  File "c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\codecarbon\core\util.py", line 24, in suppress
    yield
  File "c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\contextlib.py", line 85, in inner
    return func(*args, **kwds)
  File "c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\codecarbon\emissions_tracker.py", line 638, in stop
    self._persist_data(
    ~~~~~~~~~~~~~~~~~~^
        total_emissions=emissions_data,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        delta_emissions=emissions_data_delta,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        experiment_name=self._experiment_name,
        ^^^^^^^^^^^^^^^^^^^^^

627.5352866649628


## Plot compression metrics evolution

In [ ]:
# Compute summary statistics
avg_cpu = sum(cpu_usage) / len(cpu_usage)
peak_mem = max(mem_usage)

print(f"Average CPU usage: {avg_cpu:.2f}%")
print(f"Peak memory usage: {peak_mem:.2f} MB")
#print(f"Elapsed time: {end - starting :.2f} s")

# Calculate memory usage statistics during prediction
if mem_usage:
    min_mem = min(mem_usage)
    max_mem = max(mem_usage)
    avg_mem = sum(mem_usage) / len(mem_usage)
    print(f"Memory usage during prediction:")
    print(f"Minimum: {min_mem:.4f} MB")
    print(f"Maximum: {max_mem:.4f} MB")
    print(f"Average: {avg_mem :.4f} MB")
else:
    print("No memory usage data collected.")

In [ ]:
#create time_chain dataset with time for each ram value
start_time_str = init_time
length = len(mem_usage)
sampling_interval = 0.1
# Convert the starting time string to a datetime object
start_time = datetime.datetime.strptime(start_time_str, '%Y-%m-%d %H:%M:%S.%f')
# Create a list of timestamps based on the starting time, length, and sampling interval
time_chain = [start_time + datetime.timedelta(seconds=i * sampling_interval) for i in range(length)]
time_chain_str = [time.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] for time in time_chain]

time_chain_df = pd.DataFrame({'Timestamp': time_chain_str, 'Memory usage': mem_usage, 'CPU usage':cpu_usage })

In [ ]:

time_objects = [datetime.datetime.strptime(ts, '%Y-%m-%d %H:%M:%S.%f') for ts in times]
events_df=pd.DataFrame({'Timestamp': time_objects, 'event': [f for f in os.listdir(folder_audio) if f.endswith(".WAV") or f.endswith(".wav")]})


time_chain_df['Timestamp'] = pd.to_datetime(time_chain_df['Timestamp'])
events_df['Timestamp'] = pd.to_datetime(events_df['Timestamp'])
merged_df=pd.merge(time_chain_df, events_df, on='Timestamp', how='left', sort=True)
merged_df.to_csv(f'{folder_tracking}/Ram_usage_{method_compression}_{parameter_compression}.csv')

In [ ]:
save_path=f'{folder_tracking}/time_execution_{method_compression}_{parameter_compression}.txt'
with open(save_path, "w") as f:
    f.write(f"time execution: {execution_time_baseline}")